# NB02 — Financial Data
**Machine Learning in Finance · Week 2**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB02_financial_data.ipynb)

**Finance question:** What does the data a quant fund works with actually look like — and what's wrong with it?

**Key idea:** We model **returns, not prices**. And data can lie in two classic ways: **survivorship bias** (only the winners are left) and **look-ahead bias** (using information before it was public).

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## The course dataset
Every stock that was in the S&P 500 at any time from 2010 to 2025: prices (Yahoo Finance), financial
statements (SEC filings), market factors (Kenneth French's data library), economic data (Federal
Reserve) and financial news sentences. All free and public. Let's look at the list of companies.

In [ ]:
universe = mlfin.load_universe()
universe[["ticker", "name", "sector", "first_date", "last_date", "in_index_at_end", "has_prices"]].fillna("—").head(10)

In [ ]:
print(f"{len(universe)} stocks were in the S&P 500 at some point in 2010-2025.")
print(f"{universe['in_index_at_end'].sum()} were still in it at the end of 2025;",
      f"{(~universe['in_index_at_end']).sum()} had left.")
print(f"We have prices for {universe['has_prices'].sum()} of them.")

*Notice:* we have prices for almost every stock still in the index, but for fewer than a third of
those that left (rows with "—" are companies we know only by their old ticker). Companies that were bought or went bankrupt disappear from free data sources. Keep this
in mind — it matters at Step 5.

## Step 1: prices vs returns
A $500 stock is not "better" than a $50 stock. What investors care about is **return**: the percentage
change in value. Returns are comparable across stocks and over time, so every model in this course
predicts returns, not prices.

In [ ]:
prices = mlfin.load_prices("M")
returns = mlfin.to_returns(prices)     # (this month's price / last month's price) - 1
returns[["AAPL", "KO", "SPY"]].tail()

In [ ]:
# 🟡 YOUR TURN
TICKER = "MSFT"   # ← try "NVDA", "KO", "XOM"

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
top.plot(prices.index, prices[TICKER])
top.set_title(f"{TICKER}: month-end price")
top.set_ylabel("Price ($)")
bottom.bar(returns.index, returns[TICKER] * 100, width=20)
bottom.set_title(f"{TICKER}: monthly return")
bottom.set_ylabel("Return (%)")
bottom.set_xlabel("Date")
plt.tight_layout()
plt.show()

*What to look for:* the price chart trends and drifts; the return chart bounces around zero with
no obvious trend. Models work much better with the second kind of series.

## Step 2: how are returns distributed?
Finance textbooks often assume returns follow a bell curve (a *normal distribution*). Do they? Daily
returns give us over 4,000 observations per stock, enough to see the shape clearly.

In [ ]:
daily_returns = mlfin.to_returns(mlfin.load_prices("D"))

In [ ]:
# 🟡 YOUR TURN
HIST_TICKER = "SPY"   # ← try "TSLA" or "KO"
mlfin.plot_histogram(daily_returns[HIST_TICKER], period="Daily", bins=120,
                     title=f"Daily returns of {HIST_TICKER}, 2009-2025")

*What to look for:* the middle bars rise **above** the red curve (lots of quiet days), and the line printed
under the chart counts the extreme days — far out where the curve is almost zero (like March 2020). Extreme
days happen many times more often than a bell curve predicts. These **fat tails** are why risk models built on
the bell curve underestimate crashes.

## Step 3: compare stocks of your choice

In [ ]:
# 🟡 YOUR TURN
MY_TICKERS = ["AAPL", "KO", "JPM"]   # ← pick any three tickers

r = returns.loc["2010":, MY_TICKERS]
summary = pd.DataFrame({
    "average monthly return (%)": r.mean() * 100,
    "volatility (% per month)": r.std() * 100,
    "worst month (%)": r.min() * 100,
    "worst month was": r.idxmin().dt.strftime("%b %Y")})
summary.round(2)

## Step 4: the market as a whole
How did $1 grow in the S&P 500 (SPY) compared with an equal amount in every stock we have data for?

In [ ]:
r = returns.loc["2010":]
ew = r.drop(columns="SPY").mean(axis=1)        # average of all stocks each month
growth = pd.DataFrame({"S&P 500 (SPY)": (1 + r["SPY"]).cumprod(),
                       "Equal amount in every stock": (1 + ew).cumprod()})
growth.plot(title="Growth of $1 since 2010", ylabel="Value of $1", xlabel="Date")
plt.show()

*Hold on.* The "every stock" line looks impressive. But which stocks are in our data? Mostly ones
that **survived** to 2025…

## Step 5: survivorship bias
Suppose in 2010 you had bought **today's** S&P 500 members. That's impossible — in 2010 nobody knew
which companies would still be around in 2025. Compare it with buying the stocks that were **actually
in the index at the time**, month by month.

In [ ]:
surv = mlfin.survivorship_portfolios()
(1 + surv).cumprod().plot(title="Survivorship bias: two ways to 'own the S&P 500', 2010-2025",
                          ylabel="Value of $1", xlabel="Date")
plt.show()
years = len(surv) / 12
print(((1 + surv).prod() ** (1 / years) - 1).mul(100).round(1).rename("average yearly return (%)"))

*What to look for:* "today's members" earn about **4 percentage points a year more**. None of that
is skill — it's hindsight: the losers were quietly removed from the list. Any backtest that starts from
today's list of companies is inflated this way.
Even "members at the time" is still a bit flattering: firms that disappeared entirely aren't in our free data.

## Step 6: look-ahead bias
Companies report their financial results **weeks after** the quarter ends. Apple's quarter ending in
late December is filed with the SEC around late January. A model that used December's numbers on
December 31st would be using information nobody had yet.

In [ ]:
fund = mlfin.load_fundamentals()
apple = fund[fund["ticker"] == "AAPL"][["period_end", "report_date", "revenue", "net_income"]].tail(4).copy()
apple["days until public"] = (apple["report_date"] - apple["period_end"]).dt.days
apple

Our course tools always line up a number with its **report date**, never its period end. In Week 3
you'll see what happens when someone gets this wrong.

## Step 7: data detective
Here is a small price file of the kind you might download yourself. It has **two problems** hidden in it.
Find them.

In [ ]:
raw = mlfin.messy_prices()
raw.head()

In [ ]:
# 🟡 YOUR TURN
DETECTIVE_TICKER = "AAPL"   # ← check each of: "AAPL", "KO", "JPM", "XOM", "NKE"

raw[DETECTIVE_TICKER].plot(title=f"Raw month-end price: {DETECTIVE_TICKER}", ylabel="Price ($)", xlabel="Date")
plt.show()
moves = raw[DETECTIVE_TICKER].pct_change() * 100
print("Three biggest monthly drops (%):")
print(moves.nsmallest(3).round(1))
print("Months with no price:", list(raw.index[raw[DETECTIVE_TICKER].isna()].strftime("%b %Y")))

**Write down** what you found and what you think caused each problem (hint: look up what happened to
Apple's share count in August 2020). How would each problem fool a model?

## ⭐ GOING FURTHER (optional)
Ask an AI assistant: *"How do I find every month in a pandas DataFrame where any column fell by more
than 40%?"* Try its code on `raw` below. Did it find the AAPL problem? Did it flag anything that was real?

In [ ]:
# your AI-assisted code here

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*